
# Experiment: TransUNet Data and Weight Setup on Google Drive

Objective:
- Prepare the Synapse dataset and the R50-ViT-B/16 pretrained checkpoint on Google Drive for later TransUNet runs on Colab.
- Keep the setup notebook simple and focused on data/bootstrap only.

Notes:
- This notebook mounts `MyDrive`, downloads the dataset archive, extracts the final Synapse layout, downloads the pretrained weight, and prints the config block for the main TransUNet research notebook.
- Use this notebook once before running the main attention experiment notebook.

In [ ]:

from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive")

SYNAPSE_FOLDER_URL = "https://drive.google.com/drive/folders/1ACJEoTp-uqfFJ73qS3eUObQh52nGuzCd"
SYNAPSE_ARCHIVE_FILE_ID = "1BvpY0g9mKkkhdHpAX1HqDw8iTJNbFuwq"
SYNAPSE_ARCHIVE_NAME = "project_TransUNet.zip"
DATASET_DRIVE_DIR = DRIVE_ROOT / "datasets" / "Synapse"

WEIGHT_DRIVE_DIR = DRIVE_ROOT / "transunet"
WEIGHT_DRIVE_FILE = WEIGHT_DRIVE_DIR / "R50+ViT-B_16.npz"
WEIGHT_DOWNLOAD_URLS = [
    "https://huggingface.co/kenton-li/nnSAM/resolve/main/R50%2BViT-B_16.npz?download=true",
    "https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz",
    "https://storage.googleapis.com/vit_models/imagenet21k/R50-ViT-B_16.npz",
]

OVERWRITE_DATASET = False
OVERWRITE_WEIGHT = False
FORCE_REINSTALL_PACKAGES = True
DOWNLOAD_RETRIES = 6

print({
    "dataset_drive_dir": str(DATASET_DRIVE_DIR),
    "dataset_archive_file_id": SYNAPSE_ARCHIVE_FILE_ID,
    "weight_drive_file": str(WEIGHT_DRIVE_FILE),
    "overwrite_dataset": OVERWRITE_DATASET,
    "overwrite_weight": OVERWRITE_WEIGHT,
})

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

USE_GOOGLE_DRIVE = globals().get("USE_GOOGLE_DRIVE", True)
FORCE_REINSTALL_PACKAGES = globals().get("FORCE_REINSTALL_PACKAGES", True)

def run_cmd(cmd):
    print("$", " ".join(shlex.quote(str(part)) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

pip_install_args = ["--upgrade"]
if FORCE_REINSTALL_PACKAGES:
    pip_install_args.append("--force-reinstall")

run_cmd([sys.executable, "-m", "pip", "install", *pip_install_args, "pip", "setuptools", "wheel"])
run_cmd([sys.executable, "-m", "pip", "install", *pip_install_args, "gdown", "requests", "lxml"])


In [ ]:

from pathlib import Path
import json
import shutil
import time
import zipfile

import gdown

DRIVE_ROOT = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive"))
SYNAPSE_ARCHIVE_FILE_ID = globals().get("SYNAPSE_ARCHIVE_FILE_ID", "1BvpY0g9mKkkhdHpAX1HqDw8iTJNbFuwq")
SYNAPSE_ARCHIVE_NAME = globals().get("SYNAPSE_ARCHIVE_NAME", "project_TransUNet.zip")
DATASET_DRIVE_DIR = globals().get("DATASET_DRIVE_DIR", DRIVE_ROOT / "datasets" / "Synapse")
OVERWRITE_DATASET = globals().get("OVERWRITE_DATASET", False)
DOWNLOAD_RETRIES = globals().get("DOWNLOAD_RETRIES", 6)

def is_synapse_root(candidate: Path) -> bool:
    return (candidate / "train_npz").exists() and (candidate / "test_vol_h5").exists()

def discover_synapse_roots(search_root: Path, limit: int = 50):
    matches = []
    seen = set()
    if not search_root.exists():
        return matches
    for train_dir in search_root.rglob("train_npz"):
        root = train_dir.parent
        if is_synapse_root(root):
            key = str(root.resolve())
            if key not in seen:
                matches.append(root)
                seen.add(key)
        if len(matches) >= limit:
            break
    return sorted(matches, key=lambda p: (len(p.parts), str(p)))

def download_file_to_path(file_id: str, output_path: Path, retries: int = 6):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading file id={file_id} to {output_path} (attempt {attempt}/{retries})")
            result = gdown.download(id=file_id, output=str(output_path), quiet=False, resume=True)
            if result:
                return result
        except Exception as exc:
            wait_seconds = min(60, 10 * attempt)
            print(f"  Download failed: {exc}. Retrying in {wait_seconds}s")
            time.sleep(wait_seconds)
    raise RuntimeError(f"Failed to download file id={file_id} after {retries} attempts.")

def extract_archive_to_dataset_root(archive_path: Path, target_root: Path):
    extract_root = target_root.parent / (target_root.name + "_extract_tmp")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    print("Extracting archive to:", extract_root)
    with zipfile.ZipFile(archive_path, "r") as zip_file:
        zip_file.extractall(extract_root)

    candidates = discover_synapse_roots(extract_root, limit=100)
    if not candidates:
        raise RuntimeError(
            f"Archive extracted under {extract_root}, but no Synapse layout was found."
        )

    chosen = candidates[0]
    print("Normalizing extracted dataset from:", chosen)

    if target_root.exists():
        shutil.rmtree(target_root)
    target_root.mkdir(parents=True, exist_ok=True)

    for folder_name in ("train_npz", "test_vol_h5"):
        src = chosen / folder_name
        dst = target_root / folder_name
        shutil.move(str(src), str(dst))

    shutil.rmtree(extract_root)
    return target_root

archive_path = DATASET_DRIVE_DIR.parent / SYNAPSE_ARCHIVE_NAME

if DATASET_DRIVE_DIR.exists() and OVERWRITE_DATASET:
    print("Removing existing dataset cache:", DATASET_DRIVE_DIR)
    shutil.rmtree(DATASET_DRIVE_DIR)

if archive_path.exists() and OVERWRITE_DATASET:
    print("Removing existing dataset archive:", archive_path)
    archive_path.unlink()

if not is_synapse_root(DATASET_DRIVE_DIR):
    print("Downloading Synapse dataset archive to Drive:", archive_path)
    download_file_to_path(SYNAPSE_ARCHIVE_FILE_ID, archive_path, retries=DOWNLOAD_RETRIES)
    resolved_dataset_root = extract_archive_to_dataset_root(archive_path, DATASET_DRIVE_DIR)
else:
    print("Dataset already cached on Drive:", DATASET_DRIVE_DIR)
    resolved_dataset_root = DATASET_DRIVE_DIR

train_count = len(list((resolved_dataset_root / "train_npz").glob("*.npz"))) if (resolved_dataset_root / "train_npz").exists() else 0
test_count = len(list((resolved_dataset_root / "test_vol_h5").glob("*.npy.h5"))) if (resolved_dataset_root / "test_vol_h5").exists() else 0

print(json.dumps({
    "dataset_archive_path": str(archive_path),
    "dataset_drive_dir": str(DATASET_DRIVE_DIR),
    "resolved_dataset_root": str(resolved_dataset_root),
    "train_npz_count": train_count,
    "test_volume_count": test_count,
}, indent=2))

if train_count == 0 or test_count == 0:
    raise RuntimeError(
        "Dataset cache completed but the expected Synapse layout is still incomplete on Drive. "
        "Inspect the extracted Drive folder contents before continuing."
    )

In [ ]:
from pathlib import Path
import json
import shutil
import urllib.request

DRIVE_ROOT = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive"))
WEIGHT_DRIVE_DIR = globals().get("WEIGHT_DRIVE_DIR", DRIVE_ROOT / "transunet")
WEIGHT_DRIVE_FILE = globals().get("WEIGHT_DRIVE_FILE", WEIGHT_DRIVE_DIR / "R50+ViT-B_16.npz")
WEIGHT_DOWNLOAD_URLS = globals().get("WEIGHT_DOWNLOAD_URLS", [
    "https://huggingface.co/kenton-li/nnSAM/resolve/main/R50%2BViT-B_16.npz?download=true",
    "https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz",
    "https://storage.googleapis.com/vit_models/imagenet21k/R50-ViT-B_16.npz",
])
OVERWRITE_WEIGHT = globals().get("OVERWRITE_WEIGHT", False)

def normalize_weight_files(weight_dir: Path):
    plus_name = weight_dir / "R50+ViT-B_16.npz"
    minus_name = weight_dir / "R50-ViT-B_16.npz"
    if plus_name.exists() and not minus_name.exists():
        shutil.copy2(plus_name, minus_name)
    if minus_name.exists() and not plus_name.exists():
        shutil.copy2(minus_name, plus_name)
    if not plus_name.exists() or not minus_name.exists():
        raise FileNotFoundError(
            f"Expected both aliases in {weight_dir}, but found plus={plus_name.exists()} minus={minus_name.exists()}"
        )
    return plus_name, minus_name

def download_weight(target_file: Path, urls):
    attempted = []
    for url in urls:
        try:
            print("Trying weight URL:", url)
            urllib.request.urlretrieve(url, target_file)
            size_mb = target_file.stat().st_size / (1024 ** 2)
            print(f"Downloaded {target_file.name}: {size_mb:.1f} MB")
            if size_mb < 100:
                raise RuntimeError(f"Downloaded file too small: {size_mb:.1f} MB")
            return url
        except Exception as exc:
            attempted.append({"url": url, "error": str(exc)})
            print("  Failed:", exc)
            if target_file.exists():
                target_file.unlink()
    raise RuntimeError(json.dumps(attempted, indent=2))

if WEIGHT_DRIVE_FILE.exists() and not OVERWRITE_WEIGHT:
    print("Weight already cached on Drive:", WEIGHT_DRIVE_FILE)
else:
    WEIGHT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    used_url = download_weight(WEIGHT_DRIVE_FILE, WEIGHT_DOWNLOAD_URLS)
    print("Weight source URL selected:", used_url)

plus_weight, minus_weight = normalize_weight_files(WEIGHT_DRIVE_DIR)

print(json.dumps({
    "weight_plus": str(plus_weight),
    "weight_minus": str(minus_weight),
    "weight_size_mb": round(plus_weight.stat().st_size / (1024 ** 2), 1),
}, indent=2))


In [ ]:
from pathlib import Path

DRIVE_ROOT = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive"))
DATASET_DRIVE_DIR = globals().get("DATASET_DRIVE_DIR", DRIVE_ROOT / "datasets" / "Synapse")
WEIGHT_DRIVE_FILE = globals().get("WEIGHT_DRIVE_FILE", DRIVE_ROOT / "transunet" / "R50+ViT-B_16.npz")

config_block = f"""
DATA_SOURCE = \"drive\"
DRIVE_DATASET_DIR = Path(\"{DATASET_DRIVE_DIR}\")
COPY_DATA_TO_RUNTIME = True

WEIGHTS_SOURCE = \"drive\"
DRIVE_WEIGHT_FILE = Path(\"{WEIGHT_DRIVE_FILE}\")
"""

print("Use this config in the main training notebook:")
print(config_block)


## Next Steps

- Run the config cell, then the mount/install cell, then the dataset cache cell.
- Wait for the dataset cache to finish; it is much larger than the weight download.
- Copy the printed config block into the main training notebook.
